# Reviewer: orientation and direction tuning of depth-tuned neurons

> How does depth sensitivity relate to other sensitivities, such as orientation? Are these multiplexed,
> or are they separate sub-populations?

Six figure sessions (mice PZAH10.2d and PZAH10.2f) have drifting-grating recordings (`SFTF`) after the
depth protocol. The gratings ran as separate recordings, so no stationary-epoch selection is needed:
all grating trials are used. ROIs are the same suite2p ROIs as the depth recordings.

- **Stimulus**: full-field drifting gratings, 6 SF (0.01-0.32 cpd) × 6 TF (0.5-16 Hz) × 8 directions,
  ~4 s each with no blank, each recording showing the full grid once (5 recordings per session).
- **Response**: mean dF/F (`ast_neuropil=False` traces, as in the manuscript) over the trial, shifted by
  `LAG` imaging volumes.
- **Tuning**: for each ROI, the preferred SF/TF is the combination with the largest direction-averaged
  response; the direction tuning curve is taken at that SF/TF. Preferred direction and orientation, gDSI
  and gOSI come from the vector sum of the rectified curve. Orientation is the direction of motion mod 180°.
- **Significance**: *responsive* = one-way ANOVA across all 288 conditions; *orientation / direction
  selective* = responsive and gOSI / gDSI above a shuffle of direction labels among the trials of the
  preferred SF/TF. Shuffling within an SF/TF leaves its direction-averaged response unchanged, so the
  choice of preferred SF/TF does not bias the test.
- **Parametric fit** (summary figure): `neurons_df_sftf.pickle` of each session, from the pipeline
  (`depth_SFTF`, rerun in October 2026 with `ast_neuropil=False` dF/F and leave-one-recording-out
  cross-validation). *SF/TF tuned* = one-sided Spearman p < 0.05 between held-out responses and
  predictions, the same test as for depth tuning.
- **Depth tuning**: `depth_tuned` from `figure_utils.rf_distance.load_session_neurons` (one-sided Spearman
  on the closed-loop dF/F depth tuning, as in the figures). Only `iscell` ROIs are kept.

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import flexiznam as flz

from cottage_analysis.preprocessing import synchronisation
from cottage_analysis.plotting import style
from cottage_analysis.plotting.style import CM
from v1_depth_map.figure_utils.rf_distance import load_session_neurons
from v1_depth_map.paths import get_figures_roots

style.setup_figure_fonts()

PROJECT = "hey2_3d-vision_foodres_20220101"
SESSIONS = [
    "PZAH10.2d_S20230920",
    "PZAH10.2d_S20230922",
    "PZAH10.2f_S20230817",
    "PZAH10.2f_S20230912",
    "PZAH10.2f_S20230914",
    "PZAH10.2f_S20230924",
]
PHOTODIODE_PROTOCOL = 5  # all PZAH mice
TRACES = {"anatomical_only": 3, "ast_neuropil": False}
PARAMS = ["SpatialFrequency", "TemporalFrequency", "Angle"]
LAG = 2  # imaging volumes between stimulus and response window (as in gratings.generate_trials_df)
MIN_VOLUMES = 10  # drop trials shorter than this (truncated at the end of a recording)
P_RESPONSIVE = 0.01  # ANOVA across conditions
P_SELECTIVE = 0.05  # gOSI / gDSI shuffle
N_SHUFFLES = 1000
REBUILD = False  # True: re-sync the recordings instead of reading the trial cache
# selection criteria and stats of the thesis SFTF figure (yiran-phd-thesis/yh_thesis/depth/SFTF.ipynb),
# except SF/TF tuned, which is cross-validated (one-sided Spearman p < 0.05, as for depth)
# RS/OF tuned (panels B, C): rsof_test_rsq_closedloop_g2d above the empirical-null threshold of figure 2
# (mirrored negative tail, common_utils.add_rsq_significance)
RSOF_SIG_PERCENTILE = 95
DSI_THR = 0.6  # panel F: direction-selective cells only
N_BOOTS = 20000  # hierarchical bootstrap (mouse, session) of the B, C correlations (exported stats)
N_BOOTS_EXPLORATORY = 2000  # exploratory sections (not exported): p resolution 5e-4

GROUPS = {"depth tuned": "#2a78d6", "not depth tuned": "#eb6834"}

flm = flz.get_flexilims_session(PROJECT)
_, SAVE_ROOT = get_figures_roots(flm, fig_subdir="fig_reviewer_gratings")
CACHE = SAVE_ROOT / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
print(f"Saving to {SAVE_ROOT}")

## Grating trials

One row per trial, response = mean dF/F of each ROI over the trial window. Trials are split where the
parameter-log time (`stimulus_harptime`) changes, so two consecutive trials with the same parameters stay
separate. The last trial of a recording gets the median trial length. The trial tables are cached
per session.

In [ ]:
def recording_trials(recording):
    # Trial table and (n_trials, n_rois) mean dF/F of one SFTF recording.
    vs_df = synchronisation.generate_vs_df(
        recording=recording,
        photodiode_protocol=PHOTODIODE_PROTOCOL,
        flexilims_session=flm,
        project=PROJECT,
    )
    img_df = synchronisation.generate_imaging_df(
        vs_df=vs_df,
        recording=recording,
        flexilims_session=flm,
        filter_datasets=TRACES,
        return_volumes=True,
    ).reset_index(drop=True)
    dff = np.vstack(img_df.dffs.values)  # (n_volumes, n_rois)

    stim = img_df["stimulus_harptime"]
    onset = (stim.notna() & (stim != stim.shift())).values
    starts = np.flatnonzero(onset)
    ends = np.append(starts[1:], starts[-1] + int(np.median(np.diff(starts))))
    starts, ends = starts + LAG, np.minimum(ends + LAG, len(img_df))
    keep = ends - starts >= MIN_VOLUMES

    trials = img_df.loc[onset, PARAMS].reset_index(drop=True)
    trials["n_volumes"] = ends - starts
    trials["recording"] = recording["name"]
    resp = np.stack([dff[s:e].mean(axis=0) for s, e in zip(starts, ends)])
    return trials[keep].reset_index(drop=True), resp[keep]


def session_trials(session):
    # All SFTF trials of a session, cached in CACHE.
    path = CACHE / f"{session}_sftf_trials.pickle"
    if path.exists() and not REBUILD:
        return pd.read_pickle(path)
    recordings = flz.get_children(
        parent_name=session, children_datatype="recording", flexilims_session=flm
    )  # indexed by name, which the sync functions read as `recording.name`
    recordings = recordings[recordings["name"].str.contains("SFTF")].sort_index()
    out = [recording_trials(rec) for _, rec in recordings.iterrows()]
    data = dict(
        trials=pd.concat([o[0] for o in out], ignore_index=True),
        dff=np.concatenate([o[1] for o in out], axis=0),
    )
    pd.to_pickle(data, path)
    return data


data = {session: session_trials(session) for session in SESSIONS}

In [ ]:
summary = []
for session, d in data.items():
    t = d["trials"]
    reps = t.groupby(PARAMS).size()
    summary.append(
        dict(
            session=session,
            recordings=t.recording.nunique(),
            trials=len(t),
            rois=d["dff"].shape[1],
            conditions=len(reps),
            reps_min=reps.min(),
            reps_max=reps.max(),
            volumes_per_trial=t.n_volumes.median(),
        )
    )
summary = pd.DataFrame(summary)
summary

## Tuning metrics

In [ ]:
def vector_metrics(curves, theta):
    # gOSI, gDSI, preferred orientation and direction (deg) of rectified curves (..., n_dir).
    r = np.clip(curves, 0, None)
    total = r.sum(axis=-1)
    v1 = (r * np.exp(1j * theta)).sum(axis=-1)
    v2 = (r * np.exp(2j * theta)).sum(axis=-1)
    with np.errstate(invalid="ignore", divide="ignore"):
        gosi, gdsi = np.abs(v2) / total, np.abs(v1) / total
    pref_ori = np.rad2deg(np.angle(v2)) / 2 % 180
    pref_dir = np.rad2deg(np.angle(v1)) % 360
    return gosi, gdsi, pref_ori, pref_dir


def session_tuning(trials, resp, rng):
    values = [np.unique(trials[p]) for p in PARAMS]
    n_sf, n_tf, n_dir = (len(v) for v in values)
    i_sf, i_tf, i_dir = (np.searchsorted(v, trials[p]) for v, p in zip(values, PARAMS))
    i_sftf = i_sf * n_tf + i_tf
    cond = i_sftf * n_dir + i_dir
    n_cond, n_roi = n_sf * n_tf * n_dir, resp.shape[1]
    theta = np.deg2rad(values[2])

    counts = np.bincount(cond, minlength=n_cond)
    sums = np.zeros((n_cond, n_roi))
    np.add.at(sums, cond, resp)
    with np.errstate(invalid="ignore"):
        means = (sums / counts[:, None]).reshape(n_sf * n_tf, n_dir, n_roi)
    best = np.nanargmax(np.nanmean(means, axis=1), axis=0)  # preferred SF/TF per ROI
    curves = means[best, :, np.arange(n_roi)]  # (n_roi, n_dir)
    gosi, gdsi, pref_ori, pref_dir = vector_metrics(curves, theta)

    # shuffle direction labels among the trials of each ROI's preferred SF/TF
    p_osi, p_dsi = np.full(n_roi, np.nan), np.full(n_roi, np.nan)
    for k in np.unique(best):
        rois = np.flatnonzero(best == k)
        tr = np.flatnonzero(i_sftf == k)
        labels = np.tile(i_dir[tr], (N_SHUFFLES, 1))
        onehot = np.eye(n_dir)[rng.permuted(labels, axis=1)]  # (n_shuffles, n_trials, n_dir)
        onehot /= np.bincount(i_dir[tr], minlength=n_dir)
        shuffled = np.einsum("std,tr->srd", onehot, resp[np.ix_(tr, rois)])
        osi_sh, dsi_sh, _, _ = vector_metrics(shuffled, theta)
        p_osi[rois] = (1 + (osi_sh >= gosi[rois]).sum(axis=0)) / (N_SHUFFLES + 1)
        p_dsi[rois] = (1 + (dsi_sh >= gdsi[rois]).sum(axis=0)) / (N_SHUFFLES + 1)

    p_resp = stats.f_oneway(*[resp[cond == c] for c in range(n_cond)], axis=0).pvalue
    return pd.DataFrame(
        dict(
            roi=np.arange(n_roi),
            pref_sf=values[0][best // n_tf],
            pref_tf=values[1][best % n_tf],
            gosi=gosi,
            gdsi=gdsi,
            pref_ori=pref_ori,
            pref_dir=pref_dir,
            p_osi=p_osi,
            p_dsi=p_dsi,
            p_resp=p_resp,
        )
    ), curves


NEURONS_COLUMNS = [
    "roi", "iscell", "depth_tuned", "preferred_depth_closedloop", "preferred_OF_closedloop_g2d",
    "preferred_RS_closedloop_g2d", "rsof_test_rsq_closedloop_g2d", "rsof_popt_closedloop_g2d",
]


def session_neurons(session):
    # columns of load_session_neurons used here, cached (the full neurons_df is ~150-325 MB)
    path = CACHE / f"{session}_neurons_slim.pickle"
    if path.exists() and not REBUILD:
        return pd.read_pickle(path)
    neurons_df, _ = load_session_neurons(flm, session)
    slim = neurons_df[NEURONS_COLUMNS].reset_index(drop=True)
    slim.to_pickle(path)
    return slim


rng = np.random.default_rng(0)
cells, curves = [], {}
for session in SESSIONS:
    neurons_df = session_neurons(session)
    tuning, curves[session] = session_tuning(
        data[session]["trials"], data[session]["dff"], rng
    )
    assert len(tuning) == len(neurons_df), f"{session}: ROI count differs from neurons_df"
    tuning["session"] = session
    tuning["iscell"] = neurons_df.iscell.values.astype(bool)
    tuning["depth_tuned"] = neurons_df.depth_tuned.values.astype(bool)
    tuning["preferred_depth"] = neurons_df.preferred_depth_closedloop.values
    # RS/OF Gaussian fit of the depth protocol, rad/s -> deg/s
    tuning["preferred_of"] = np.degrees(neurons_df.preferred_OF_closedloop_g2d.astype(float).values)
    tuning["rsof_test_rsq"] = neurons_df.rsof_test_rsq_closedloop_g2d.astype(float).values
    tuning["preferred_rs"] = neurons_df.preferred_RS_closedloop_g2d.astype(float).values  # m/s
    cells.append(tuning)
cells = pd.concat(cells, ignore_index=True)
cells = cells[cells.iscell].reset_index(drop=True)
cells["group"] = np.where(cells.depth_tuned, "depth tuned", "not depth tuned")
cells["responsive"] = cells.p_resp < P_RESPONSIVE
cells["ori_selective"] = cells.responsive & (cells.p_osi < P_SELECTIVE)
cells["dir_selective"] = cells.responsive & (cells.p_dsi < P_SELECTIVE)
print(f"{len(cells)} cells")

## Parametric fit

`neurons_df_sftf.pickle` of each session (see `revisions/sftf/NEURONS_DF_SFTF.md`): `fit_sftf_tuning` on
single-trial responses, a 2D Gaussian in log SF × log TF times a von Mises direction tuning with a direction
selectivity parameter `dsi` (0: equal response to both directions of an orientation, 1: no response to the
opposite direction). Preferred speed = TF0 / SF0 (deg/s), preferred direction = `alpha0`, preferred
orientation = `alpha0` mod 180°. Each ROI is also fitted on 4 of the 5 recordings and evaluated on the
5th; a cell is *SF/TF tuned* if the Spearman correlation between held-out responses and predictions is
significantly positive (one-sided p < 0.05, `add_one_sided_spearman_significance`, as for depth tuning).

In [ ]:
from cottage_analysis.analysis import common_utils
from cottage_analysis.pipelines import pipeline_utils


def load_sftf_fit(session):
    folder = pipeline_utils.create_neurons_ds(
        session_name=session, flexilims_session=flm, project=PROJECT, conflicts="skip"
    ).path_full.parent
    fit = pd.read_pickle(folder / "neurons_df_sftf.pickle")
    assert "sftf_test_spearmanr_pval" in fit, f"{session}: neurons_df_sftf has no cross-validation"
    fit.insert(0, "roi", np.arange(len(fit)))
    fit.insert(0, "session", session)
    return fit.drop(columns=["sftf_test_popts"])


fits = pd.concat([load_sftf_fit(s) for s in SESSIONS], ignore_index=True)
cells = cells.drop(columns=[c for c in cells.columns if c.startswith("fit_")]).merge(
    fits.rename(columns={c: f"fit_{c}" for c in fits.columns if c not in ("session", "roi")}),
    on=["session", "roi"],
    how="left",
    validate="one_to_one",
)
cells["pref_speed"] = np.exp(cells.fit_tf0 - cells.fit_sf0)  # deg/s
cells["fit_dir"] = np.rad2deg(cells.fit_alpha0) % 360
cells["fit_ori"] = cells.fit_dir % 180
common_utils.add_one_sided_spearman_significance(
    cells,
    rval_col="fit_sftf_test_spearmanr_rval",
    pval_col="fit_sftf_test_spearmanr_pval",
    out_col="sftf_tuned",
)
rsof_thresholds = common_utils.add_rsq_significance(
    cells, {"g2d": "rsof_test_rsq"}, percentile=RSOF_SIG_PERCENTILE, verbose=False
)
cells["rsof_tuned"] = cells["rsof_test_rsq_sig"]
cells.to_csv(SAVE_ROOT / "grating_tuning_cells.csv", index=False)
print(f"SF/TF tuned (cross-validated): {cells.sftf_tuned.sum()} / {len(cells)}")
print(f"  for reference, in-sample R² > 0.05 (thesis): {(cells.fit_rsq > 0.05).sum()}")
print(f"RS/OF tuned (test R² > empirical-null threshold {rsof_thresholds}): {cells.rsof_tuned.sum()}")
print(f"SF/TF tuned and depth selective: {(cells.sftf_tuned & cells.depth_tuned).sum()}")
print(f"SF/TF tuned and RS/OF tuned: {(cells.sftf_tuned & cells.rsof_tuned).sum()}")

In [ ]:
# in-sample vs held-out fit quality
fig, axes = plt.subplots(1, 2, figsize=(12 * CM, 5 * CM))
ax = axes[0]
ax.scatter(cells.fit_rsq, cells.fit_sftf_test_rsq, s=3, color="k", alpha=0.3, linewidths=0)
lim = [min(cells.fit_sftf_test_rsq.min(), 0), cells.fit_rsq.max()]
ax.plot(lim, lim, color="#8a8984", lw=0.8, ls="--")
ax.set_xlabel("In-sample R²")
ax.set_ylabel("Held-out R²")
ax = axes[1]
ax.hist(cells.fit_sftf_test_spearmanr_rval, bins=50, color="k")
ax.set_xlabel("Held-out Spearman r")
ax.set_ylabel("Number of cells")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
style.savefig(SAVE_ROOT / "sftf_crossval.pdf", fig=fig, bbox_inches="tight")

## Summary figure

- **A**: overlap of depth-selective and SF/TF-tuned cells (areas proportional to cell counts).
- **B, C**: preferred grating speed vs preferred optic flow speed and preferred virtual depth (depth
  protocol), for cells that are SF/TF tuned and RS/OF tuned (RS/OF test R² above the empirical-null threshold,
  as in figure 2; fits on a bound are kept).
  Spearman ρ with a hierarchical bootstrap over mice and sessions (`N_BOOTS`), as in the thesis.
  Dashed line in B: identity.
- **D, E**: direction selectivity (`dsi`) and preferred orientation of SF/TF-tuned, depth-selective cells.
- **F**: preferred direction of SF/TF-tuned, depth-selective cells with `dsi` > `DSI_THR`.

In [ ]:
from scipy.optimize import brentq


def circle_overlap(r1, r2, d):
    # Area of the intersection of two circles of radii r1, r2 with centres d apart.
    if d >= r1 + r2:
        return 0.0
    if d <= abs(r1 - r2):
        return np.pi * min(r1, r2) ** 2
    a1 = r1**2 * np.arccos((d**2 + r1**2 - r2**2) / (2 * d * r1))
    a2 = r2**2 * np.arccos((d**2 + r2**2 - r1**2) / (2 * d * r2))
    return a1 + a2 - 0.5 * np.sqrt((-d + r1 + r2) * (d + r1 - r2) * (d - r1 + r2) * (d + r1 + r2))


def plot_venn(ax, n_all, n_a, n_b, n_ab, names, colors):
    # Area-proportional Venn of sets a and b inside the set of all cells.
    R, ra, rb = np.sqrt(np.array([n_all, n_a, n_b]) / np.pi)
    lo, hi = abs(ra - rb), ra + rb
    d = brentq(lambda x: circle_overlap(ra, rb, x) - n_ab, lo + 1e-9, hi - 1e-9)
    xa = -(ra + d + rb) / 2 + ra
    xb = xa + d
    # highest position keeping both circles inside the outer one
    y = 0.97 * min(np.sqrt(max((R - r) ** 2 - x**2, 0)) for r, x in [(ra, xa), (rb, xb)])
    ax.add_patch(plt.Circle((0, 0), R, color="#d9d8d4", lw=0))
    ax.add_patch(plt.Circle((xa, y), ra, color=colors[0], alpha=0.6, lw=0))
    ax.add_patch(plt.Circle((xb, y), rb, color=colors[1], alpha=0.6, lw=0))
    pct = lambda n: f"{100 * n / n_all:.0f}%"
    for x, n in [
        ((xa - ra + xb - rb) / 2, n_a - n_ab),
        ((xb - rb + xa + ra) / 2, n_ab),
        ((xa + ra + xb + rb) / 2, n_b - n_ab),
    ]:
        ax.text(x, y, pct(n), ha="center", va="center")
    ax.text(0, -0.75 * R, pct(n_all - n_a - n_b + n_ab), ha="center", va="center")
    ax.text(xa - ra, y + ra, names[0], ha="right", va="bottom")
    ax.text(xb + 0.3 * rb, y + rb, names[1], ha="left", va="bottom")
    ax.text(0, -R * 1.02, f"All neurons (n={n_all})", ha="center", va="top")
    ax.set_xlim(-R * 1.05, R * 1.05)
    ax.set_ylim(-R * 1.25, R * 1.15)
    ax.set_aspect("equal")
    ax.axis("off")


def polar_hist(ax, values_deg, period, n_bins):
    # Bars centred on multiples of period / n_bins (the stimulus directions when n_bins = 8).
    width = period / n_bins
    counts, _ = np.histogram((values_deg + width / 2) % period, bins=n_bins, range=(0, period))
    centres = np.deg2rad(np.arange(n_bins) * width)
    ax.bar(centres, counts, width=np.deg2rad(width) * 0.95, color="k", lw=0)
    if period < 360:  # the first bin wraps around: draw its other half at `period`
        ax.bar(np.deg2rad(period), counts[0], width=np.deg2rad(width) * 0.95, color="k", lw=0)
    ax.set_thetamin(0)
    ax.set_thetamax(period)
    ax.set_xticks(np.deg2rad(np.arange(0, period + 1 if period < 360 else period, 45)))
    ax.set_rlabel_position(period / n_bins / 2)  # radial labels between the first two bars
    ax.yaxis.set_major_locator(plt.MaxNLocator(3))
    ax.tick_params(labelsize=5)
    return counts


def log_scatter(ax, x, y, stat, xlabel, ylabel):
    ax.scatter(x, y, s=8, color="k", alpha=0.6, linewidths=0)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.spines[["top", "right"]].set_visible(False)
    ci = stat["ci"]
    ax.set_title(
        f"n = {stat['n']}, ρ = {stat['rho']:.2f} [{ci[0]:.2f}, {ci[1]:.2f}], bootstrap p = {stat['p']:.2g}",
        fontsize=6,
    )

In [ ]:
# summary figure statistics: cell selections, Venn counts, B and C correlations
import contextlib
import io

from cottage_analysis.analysis import common_utils


def bootstrap_spearman(x, y, sessions, n_boots=N_BOOTS):
    # Spearman rho, 95% CI and p from a hierarchical bootstrap over mice then sessions (thesis stats)
    df = pd.DataFrame(dict(x=np.asarray(x), y=np.asarray(y), session=np.asarray(sessions)))
    df = df[np.isfinite(df.x) & np.isfinite(df.y)].reset_index(drop=True)
    with contextlib.redirect_stderr(io.StringIO()):  # silence the tqdm progress bar
        rho, dist = common_utils.hierarchical_bootstrap_stats(
            df, n_boots, ["x"], ["mouse", "session"], ycol=["y"], correlation=True, plot=False
        )
    return dict(
        n=len(df),
        n_sessions=df.session.nunique(),
        rho=float(rho[0]),
        ci=np.percentile(dist[:, 0], [2.5, 97.5]),
        p=float(common_utils.calculate_pval_from_bootstrap(dist[:, 0], value=0)),
    )


tuned = cells[cells.sftf_tuned & cells.depth_tuned]  # D, E
dir_tuned = tuned[tuned.fit_dsi > DSI_THR]  # F
both = cells[cells.sftf_tuned & cells.rsof_tuned].reset_index(drop=True)  # B, C

venn = dict(
    n_all=len(cells),
    n_depth=int(cells.depth_tuned.sum()),
    n_sftf=int(cells.sftf_tuned.sum()),
    n_both=int((cells.depth_tuned & cells.sftf_tuned).sum()),
)
venn["n_both_expected"] = venn["n_depth"] * venn["n_sftf"] / venn["n_all"]  # if independent
stat_speed_of = bootstrap_spearman(both.pref_speed, both.preferred_of, both.session)
stat_speed_depth = bootstrap_spearman(both.pref_speed, both.preferred_depth * 100, both.session)

print(venn)
for name, st in [("grating speed vs preferred OF", stat_speed_of), ("grating speed vs preferred depth", stat_speed_depth)]:
    print(f"{name}: n = {st['n']}, rho = {st['rho']:.3f}, 95% CI {st['ci'].round(3)}, p = {st['p']:.3g}")
print(f"D, E: {len(tuned)} cells; F: {len(dir_tuned)} cells")

In [ ]:
fig = plt.figure(figsize=(18 * CM, 12 * CM))
axes = {
    "A": fig.add_subplot(2, 3, 1),
    "B": fig.add_subplot(2, 3, 2),
    "C": fig.add_subplot(2, 3, 3),
    "D": fig.add_subplot(2, 3, 4),
    "E": fig.add_subplot(2, 3, 5, projection="polar"),
    "F": fig.add_subplot(2, 3, 6, projection="polar"),
}
plot_venn(
    axes["A"],
    venn["n_all"],
    venn["n_depth"],
    venn["n_sftf"],
    venn["n_both"],
    names=["Depth\nselective", "SF/TF\ntuned"],
    colors=["#2a78d6", "#1baf7a"],
)
log_scatter(axes["B"], both.pref_speed, both.preferred_of, stat_speed_of,
            "Preferred grating speed (deg/s)", "Preferred optic flow speed (deg/s)")
lims = [min(axes["B"].get_xlim()[0], axes["B"].get_ylim()[0]), max(axes["B"].get_xlim()[1], axes["B"].get_ylim()[1])]
axes["B"].plot(lims, lims, color="#8a8984", lw=0.8, ls="--", zorder=0)
log_scatter(axes["C"], both.pref_speed, both.preferred_depth * 100, stat_speed_depth,
            "Preferred grating speed (deg/s)", "Preferred virtual depth (cm)")

axes["D"].hist(tuned.fit_dsi, bins=20, range=(0, 1), color="k")
axes["D"].set_xlabel("Direction selectivity index")
axes["D"].set_ylabel("Number of neurons")
axes["D"].spines[["top", "right"]].set_visible(False)

polar_hist(axes["E"], tuned.fit_ori.values, 180, 4)
axes["E"].set_title("Preferred orientation (deg)", fontsize=7)
polar_hist(axes["F"], dir_tuned.fit_dir.values, 360, 8)
axes["F"].set_title("Preferred direction (deg)", fontsize=7)

for letter, ax in axes.items():
    ax.text(-0.15, 1.08, letter, transform=ax.transAxes, fontweight="bold", fontsize=9, va="bottom")
fig.tight_layout()
style.savefig(SAVE_ROOT / "fig_sftf_summary.pdf", fig=fig, bbox_inches="tight")

## Example neurons

Layout of the thesis example figure (`figure_utils/sftf_examples.py`): depth tuning and optic flow tuning
(closed-loop depth protocol), then mean grating responses (left) and the fitted model (right) for the 8
directions around a polar plot. One depth-selective and one non-depth-selective SF/TF-tuned cell, both
`iscell`. ROI 544 is the thesis example; the thesis depth-selective example (ROI 229 of the same session)
touches the edge of the registered field of view and is not `iscell`.

In [ ]:
from v1_depth_map.figure_utils import sftf_examples

# (session, roi): y-axis rounding of the depth tuning curve
EXAMPLES = {
    ("PZAH10.2f_S20230914", 447): dict(ylim_precision_base=0.1, ylim_precision=2),  # depth selective
    ("PZAH10.2f_S20230817", 544): dict(ylim_precision_base=0.1, ylim_precision=2),  # not depth selective
}
print(cells.set_index(["session", "roi"]).loc[list(EXAMPLES),
      ["depth_tuned", "preferred_depth", "sftf_tuned", "fit_sftf_test_rsq", "fit_dsi", "fit_dir"]].round(2))

example_data = {s: sftf_examples.load_example_session(flm, s) for s in {s for s, _ in EXAMPLES}}
fig = plt.figure(figsize=(18 * CM, 18 * CM))
for i, ((session, roi), ylim) in enumerate(EXAMPLES.items()):
    sftf_examples.plot_example_neuron(
        fig, example_data[session], roi, top=1 - i * 0.65,
        fontsize_dict={"title": 7, "label": 7, "tick": 5, "legend": 5}, **ylim,
    )
style.savefig(SAVE_ROOT / "fig_sftf_examples.pdf", fig=fig, bbox_inches="tight")

## Exploratory: SF / TF responses by running speed

Mean dF/F per SF, TF and direction (data only, no fit) computed from running imaging frames (RS > 1 cm/s,
dF/F 5 frames after the stimulus, as in `precompute_data/sftf_running_models.py`), split into the treadmill
running-speed bins of figure 3 (one octave around each motor speed) with the two slowest merged: 2.8-11.3, 11.3-22.6,
22.6-45.3 and 45.3-90.5 cm/s (frames outside are dropped). Colour scale shared across the bins of a cell.
Blank squares: no frame of that condition in the bin (whole trials can fall in one speed bin). Cells: the two
example neurons, and the three with the largest held-out gain of the TF-dependent model in
`sftf_running_models` (S20230817 ROIs 244 and 214, S20230914 ROI 440).

In [ ]:
from v1_depth_map.precompute_data import sftf_running_models

SPEED_EXAMPLES = [
    ("PZAH10.2f_S20230914", 447),
    ("PZAH10.2f_S20230914", 440),
    ("PZAH10.2f_S20230817", 544),
    ("PZAH10.2f_S20230817", 244),
    ("PZAH10.2f_S20230817", 214),
]
# running-speed bins used by every plot split by speed: the treadmill bins of figure 3
# (figure_utils.treadmill.compute_treadmill_rsof_bins on PZAG17.3a_S20250402, one octave around each motor speed,
# 4, 8, 15, 30, 61 cm/s; rs_bins[2:] as in panels G-I), with the two slowest merged; frames outside are dropped
SPEED_EDGES_CM_S = [2.83, 11.31, 22.63, 45.25, 90.51]
# fewer, fuller bins: the three slowest figure 3 bins merged
SPEED_EDGES_3_CM_S = [2.83, 22.63, 45.25, 90.51]


def example_frames(session, rois):
    # running frames of a session and the dF/F of `rois` only, cached
    path = CACHE / f"{session}_speed_frames.pickle"
    if path.exists() and not REBUILD:
        cached = pd.read_pickle(path)
        if set(rois) <= set(cached["dff"]):
            return cached
    X, dff, _, log_rs_median = sftf_running_models.session_frames(flm, session)
    cached = dict(X=X, log_rs_median=log_rs_median, dff={r: dff[:, r] for r in rois})
    pd.to_pickle(cached, path)
    return cached


speed_frames = {
    s: example_frames(s, [r for s2, r in SPEED_EXAMPLES if s2 == s]) for s in {s for s, _ in SPEED_EXAMPLES}
}
speed_binned = {}
for session, roi in SPEED_EXAMPLES:
    f = speed_frames[session]
    binned = sftf_examples.speed_binned_responses(
        f["X"], f["dff"][roi], f["log_rs_median"], edges_cm_s=SPEED_EDGES_CM_S
    )
    speed_binned[(session, roi)] = binned
    fig = plt.figure(figsize=(18 * CM, 5.5 * CM))
    sftf_examples.plot_sftf_by_speed(
        fig, binned, title=f"{session} ROI {roi}",
        fontsize_dict={"title": 7, "label": 6, "tick": 5, "legend": 5},
    )
    style.savefig(SAVE_ROOT / f"sftf_by_speed_{session}_{roi}.pdf", fig=fig, bbox_inches="tight")

TF tuning per running-speed bin. For each cell, the SF / direction pairs whose mean response (over TF and all
running frames in 2.8-90.5 cm/s) is at least `TF_SELECT_THRESHOLD` of the best pair are kept (same pairs for all
bins); the curve is the mean over these pairs of the response at each TF, ± SEM across pairs. Top: the 4 speed bins
above; bottom: 3 bins (2.8-22.6, 22.6-45.3, 45.3-90.5 cm/s), more frames per bin.

In [ ]:
TF_SELECT_THRESHOLD = 0.5
# second row: 3 bins, more frames each
fig, axes = plt.subplots(2, len(SPEED_EXAMPLES), figsize=(18 * CM, 9 * CM))
for j, (session, roi) in enumerate(SPEED_EXAMPLES):
    f = speed_frames[session]
    binned_3 = sftf_examples.speed_binned_responses(
        f["X"], f["dff"][roi], f["log_rs_median"], edges_cm_s=SPEED_EDGES_3_CM_S
    )
    for i, binned in enumerate([speed_binned[(session, roi)], binned_3]):
        curves, kept = sftf_examples.tf_curves_by_speed(binned, threshold=TF_SELECT_THRESHOLD)
        sftf_examples.plot_tf_curves(axes[i, j], curves)
        if i == 0:
            axes[i, j].set_title(f"{session.split('_')[1]} ROI {roi}\n{len(kept)} SF/direction pairs", fontsize=6)
for i in range(2):
    axes[i, -1].legend(title="Running speed", fontsize=5, title_fontsize=5, frameon=False,
                       bbox_to_anchor=(1, 1), loc="upper left")
fig.tight_layout()
style.savefig(SAVE_ROOT / "tf_curves_by_speed.pdf", fig=fig, bbox_inches="tight")

Running-speed distribution during the gratings, per session: running frames (RS > 1 cm/s, the frames used above),
log-spaced bins, proportion of frames. Dotted lines: speed bin edges.

In [ ]:
rs_bins = np.logspace(np.log10(1), np.log10(300), 50)
fig, ax = plt.subplots(figsize=(9 * CM, 5 * CM))
colors = plt.get_cmap("tab10")(np.arange(len(SESSIONS)))
for session, color in zip(SESSIONS, colors):
    d = np.load(SAVE_ROOT / "running_models" / "frames" / f"{session}.npz")
    rs_cm_s = np.exp(d["X"][3] + float(d["log_rs_median"])) * 100
    counts, _ = np.histogram(rs_cm_s, bins=rs_bins)
    centres = np.sqrt(rs_bins[1:] * rs_bins[:-1])
    ax.plot(centres, counts / counts.sum(), color=color, lw=1,
            label=f"{session} (median {np.median(rs_cm_s):.0f} cm/s)")
for edge in SPEED_EDGES_CM_S:
    ax.axvline(edge, color="k", lw=0.5, ls=":")
ax.set_xscale("log")
ax.set_xlabel("Running speed (cm/s)", fontsize=6)
ax.set_ylabel("Proportion of running frames", fontsize=6)
ax.tick_params(labelsize=5)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(fontsize=5, frameon=False, bbox_to_anchor=(1, 1), loc="upper left")
style.savefig(SAVE_ROOT / "running_speed_distribution.pdf", fig=fig, bbox_inches="tight")

## Running-speed dependence of SF / TF preferences: model comparison

Frame-level fits of `precompute_data/sftf_running_models.py` (NEMO, SF/TF-tuned `iscell` cells): comparing
running-speed modulation models:
- **base**: no running-speed dependence (SF, TF grating model)
- **gain**: multiplicative gain scaling ($A \propto \text{RS}^g$)
- **tf**: temporal frequency shift ($\text{TF}_0 \propto \text{RS}^{a_\text{tf}}$)
- **gain_tf**: combined multiplicative gain and temporal frequency shift
- **sf**: spatial frequency shift ($\text{SF}_0 \propto \text{RS}^{a_\text{sf}}$)
- **both**: combined spatial and temporal frequency shifts

Running frames only (RS > 1 cm/s), dF/F 5 frames after stimulus; held-out R² leaving one recording out.
Cells are restricted, as in figure 2E, to those whose held-out R² is above the empirical-null threshold
(mirrored negative tail, 95th percentile, one threshold per model) for at least one of the candidate models.

In [ ]:
# running-speed model comparison statistics
RS_MODELS = ["base", "gain", "tf", "gain_tf", "sf", "both"]
pickle_files = [f for f in sorted((SAVE_ROOT / "running_models").glob("*_sftf_running_models.pickle")) if not f.name.startswith(".")]
rs_fits = pd.concat([pd.read_pickle(f) for f in pickle_files], ignore_index=True)
rs_fits = rs_fits.merge(cells[["session", "roi", "depth_tuned"]], on=["session", "roi"], how="left", validate="one_to_one")
rs_fits["best_model"] = rs_fits[[f"{m}_test_rsq" for m in RS_MODELS]].idxmax(axis=1).str.replace("_test_rsq", "")
for m in RS_MODELS[1:]:
    rs_fits[f"delta_{m}"] = rs_fits[f"{m}_test_rsq"] - rs_fits.base_test_rsq
rs_thresholds = common_utils.add_rsq_significance(
    rs_fits, {m: f"{m}_test_rsq" for m in RS_MODELS}, verbose=False
)
rs_fits["any_sig"] = rs_fits[[f"{m}_test_rsq_sig" for m in RS_MODELS]].any(axis=1)


def bootstrap_median(values, sessions, n_boots=N_BOOTS_EXPLORATORY):
    # median, 95% CI and p (vs 0) from a hierarchical bootstrap over mice then sessions
    df = pd.DataFrame(dict(x=np.asarray(values), session=np.asarray(sessions))).dropna().reset_index(drop=True)
    with contextlib.redirect_stderr(io.StringIO()):
        _, dist = common_utils.hierarchical_bootstrap_stats(df, n_boots, ["x"], ["mouse", "session"], plot=False)
    return dict(n=len(df), median=float(np.median(df.x)), ci=np.percentile(dist[:, 0], [2.5, 97.5]),
                p=float(common_utils.calculate_pval_from_bootstrap(dist[:, 0], value=0)),
                frac_pos=float(np.mean(df.x > 0)))


rs_sets = {"all": rs_fits, "selected": rs_fits[rs_fits.any_sig]}
print(f"{len(rs_fits)} cells, {rs_fits.any_sig.sum()} significant in at least one model "
      f"(thresholds {({m: round(float(v), 4) for m, v in rs_thresholds.items()})})")
best_props = pd.DataFrame({k: v.best_model.value_counts(normalize=True).reindex(RS_MODELS) for k, v in rs_sets.items()})
for k, v in rs_sets.items():
    for g, gv in v.groupby("depth_tuned"):
        best_props[f"{k}, {'depth' if g else 'not depth'} selective"] = gv.best_model.value_counts(normalize=True).reindex(RS_MODELS)
display(best_props.round(3))

rs_stats = {}
cols_to_test = [
    "gain_g", "gain_tf_g", "tf_a_tf", "gain_tf_a_tf", "sf_a_sf", "both_a_tf", "both_a_sf",
    "delta_gain", "delta_tf", "delta_gain_tf", "delta_sf", "delta_both"
]
for k, v in rs_sets.items():
    for col in cols_to_test:
        rs_stats[(k, col)] = bootstrap_median(v[col], v.session)
rs_stats_table = pd.DataFrame(
    {f"{k}: {col}": dict(n=st["n"], median=st["median"], ci_low=st["ci"][0], ci_high=st["ci"][1],
                         p=st["p"], frac_positive=st["frac_pos"]) for (k, col), st in rs_stats.items()}
).T
display(rs_stats_table.round(5))
for k in rs_sets:
    a_pure = rs_stats[(k, "tf_a_tf")]["median"]
    a_gain = rs_stats[(k, "gain_tf_a_tf")]["median"]
    print(f"{k}: pure TF slope {a_pure:.3f} -> x{2 ** a_pure:.2f} per doubling RS; with gain {a_gain:.3f} -> x{2 ** a_gain:.2f}")


In [ ]:
import seaborn as sns
from cottage_analysis.plotting import rsof_plots

FONTSIZE_DICT_SMALL = {"title": 6, "label": 6, "tick": 5}


def fmt_boot_p(p, n_boots=N_BOOTS_EXPLORATORY):
    return f"p < {1 / n_boots:.0e}" if p == 0 else f"p = {p:.2g}"


fig, axes = plt.subplots(1, 4, figsize=(22 * CM, 6 * CM), gridspec_kw=dict(width_ratios=[0.9, 0.9, 1.1, 1.1]))
fig.tight_layout()

# A: proportion of cells best explained by each model, per session (as figure 2E), depth selective or not
RS_MODEL_LABELS = ["SF, TF", "Gain", "TF shift", "Gain + TF", "SF shift", "SF+TF shift"]
rs_best_props = {}
for ax, (name, is_depth) in zip(axes[:2], [("depth selective", True), ("not depth selective", False)]):
    pos = ax.get_position()
    ax.remove()
    sel = rs_sets["selected"][rs_sets["selected"].depth_tuned == is_depth].assign(iscell=1)
    sel = sel.rename(columns={f"{m}_test_rsq": f"rsof_test_rsq_closedloop_{m}" for m in RS_MODELS})
    rs_best_props[name] = rsof_plots.plot_r2_comparison(
        fig=fig, neurons_df=sel, models=RS_MODELS, labels=RS_MODEL_LABELS, plot_type="bar", markersize=4,
        alpha=0.5, color="grey", plot_x=pos.x0, plot_y=pos.y0, plot_width=pos.width, plot_height=pos.height,
        reference_model="base", fontsize_dict=FONTSIZE_DICT_SMALL,
    )
    ax_a = plt.gca()
    ax_a.set_ylim([-0.02, 1.02])
    sns.despine(ax=ax_a, offset=5, trim=True)
    ax_a.tick_params(axis="x", rotation=90)
    ax_a.set_title(f"{name}\nn = {len(sel)} cells, {sel.session.nunique()} sessions", fontsize=6)
    if not is_depth:
        ax_a.set_ylabel("")

# B: parameter distributions (selected cells)
ax = axes[2]
v = rs_sets["selected"]
bins = np.linspace(-1, 1.5, 41)
colors_b = {"gain_g": "#e6550d", "gain_tf_a_tf": "#3182bd", "tf_a_tf": "#9ecae1"}
labels_b = {"gain_g": "Gain g", "gain_tf_a_tf": "TF a_tf (with gain)", "tf_a_tf": "TF a_tf (no gain)"}
for col in ["gain_g", "tf_a_tf", "gain_tf_a_tf"]:
    st = rs_stats[("selected", col)]
    med_val = st["median"]
    p_val = fmt_boot_p(st["p"])
    lbl = labels_b[col]
    ax.hist(np.clip(v[col], bins[0], bins[-1]), bins=bins, histtype="step", color=colors_b[col], lw=1.2,
            label=f"{lbl}: med {med_val:.2f}, {p_val}")
    ax.axvline(med_val, color=colors_b[col], lw=0.8, ls="--")
ax.axvline(0, color="k", lw=0.5)
ax.set_xlabel("Parameter slope")
ax.set_ylabel("Number of cells")
ax.legend(frameon=False, fontsize=4.5)

# C: held-out R² gain over the base model (selected cells)
ax = axes[3]
colors_c = {"delta_gain": "#e6550d", "delta_tf": "#9ecae1", "delta_gain_tf": "#3182bd", "delta_sf": "#756bb1"}
labels_c = {"delta_gain": "Gain", "delta_tf": "TF shift", "delta_gain_tf": "Gain + TF", "delta_sf": "SF shift"}
for m in ["delta_gain", "delta_tf", "delta_gain_tf", "delta_sf"]:
    g = np.sort(v[m].dropna())
    st = rs_stats[("selected", m)]
    med_val = st["median"]
    lbl = labels_c[m]
    ax.plot(g, np.arange(1, len(g) + 1) / len(g), color=colors_c[m], lw=1.2,
            label=f"{lbl}: med {med_val:.1e}")
ax.axvline(0, color="k", lw=0.5)
ax.set_xlim([-0.005, 0.015])
ax.set_xlabel("Held-out R² gain over base")
ax.set_ylabel("Cumulative proportion")
ax.legend(frameon=False, fontsize=4.5)
for ax in axes[2:]:
    ax.spines[["top", "right"]].set_visible(False)

style.savefig(SAVE_ROOT / "running_speed_models.pdf", fig=fig, bbox_inches="tight")
print(f"Saved to {SAVE_ROOT / 'running_speed_models.pdf'}")


### Hypothesis: all cells are gain modulated, depth-selective cells also shift their TF preference

Based on figure 3. In the `gain_tf` model (gain `g`: amplitude ∝ RS^g, and TF shift `a_tf`: TF0 ∝ RS^a_tf, fitted
together), compare depth-selective and other cells among the selected cells (significant in at least one model):
the gain `g` (expected in all cells), the TF shift `a_tf`, the held-out R² gain of `gain_tf` over `gain` (does the
TF shift generalise), and how often a TF-shift model (`tf` or `gain_tf`) is the best one. Medians (fractions for the
best model) compared with permutations of the depth labels within session, and within session × base-fit-quality
tertile (depth-selective cells could simply be better fitted); 10000 permutations. Values against 0 within each group:
Wilcoxon signed-rank (cells) and hierarchical bootstrap (mice, sessions).

In [ ]:
# depth-selective vs other cells: gain and TF shift
from scipy import stats as sstats

hyp = rs_sets["selected"].copy()
hyp["best"] = hyp[[f"{m}_test_rsq" for m in RS_MODELS]].idxmax(axis=1).str.replace("_test_rsq", "")
hyp["shift_best"] = hyp.best.isin(["tf", "gain_tf"])
hyp["dR2_shift"] = hyp.gain_tf_test_rsq - hyp.gain_test_rsq
hyp["fit_tertile"] = pd.qcut(hyp.base_test_rsq, 3, labels=["low", "mid", "high"]).astype(str)
N_PERM = 10000
perm_rng = np.random.default_rng(0)


def depth_permutation(df, col, stat, strata, n=N_PERM):
    # difference (depth - other) of `stat` and one-sided p, depth labels permuted within `strata`
    vals, labels = df[col].to_numpy(float), df.depth_tuned.to_numpy(bool)
    obs = stat(vals[labels]) - stat(vals[~labels])
    groups = [np.flatnonzero(df[strata].agg("_".join, axis=1).to_numpy() == g)
              for g in df[strata].agg("_".join, axis=1).unique()]
    null = np.empty(n)
    for i in range(n):
        lab = labels.copy()
        for idx in groups:
            lab[idx] = perm_rng.permutation(lab[idx])
        null[i] = stat(vals[lab]) - stat(vals[~lab])
    return obs, (1 + np.sum(null >= obs)) / (n + 1)


HYP_MEASURES = {  # column: (label, statistic)
    "gain_tf_g": ("gain g (gain_tf)", np.median),
    "gain_tf_a_tf": ("TF shift a_tf (gain_tf)", np.median),
    "dR2_shift": ("held-out R² gain_tf - gain", np.median),
    "shift_best": ("best model tf or gain_tf", np.mean),
}
rows = []
for col, (label, stat) in HYP_MEASURES.items():
    obs, p_s = depth_permutation(hyp, col, stat, ["session"])
    _, p_sq = depth_permutation(hyp, col, stat, ["session", "fit_tertile"])
    row = dict(measure=label, depth=stat(hyp[col][hyp.depth_tuned]), other=stat(hyp[col][~hyp.depth_tuned]),
               diff=obs, p_within_session=p_s, p_within_session_fit=p_sq)
    if col != "shift_best":
        for name, is_depth in [("depth", True), ("other", False)]:
            v = hyp[col][hyp.depth_tuned == is_depth]
            row[f"{name}_wilcoxon_p"] = sstats.wilcoxon(v).pvalue
            row[f"{name}_boot_p"] = bootstrap_median(v, hyp.session[hyp.depth_tuned == is_depth])["p"]
    rows.append(row)
hyp_table = pd.DataFrame(rows).set_index("measure")
print(f"{hyp.depth_tuned.sum()} depth-selective and {(~hyp.depth_tuned).sum()} other selected cells; "
      "p_within_*: one-sided (depth > other)")
display(hyp_table.round(4))
display(hyp.groupby(["session", "depth_tuned"])[["gain_tf_g", "gain_tf_a_tf"]].median().unstack().round(3))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18 * CM, 5 * CM))
for ax, (col, xlabel, xlim) in zip(axes, [
    ("gain_tf_g", "Gain g (amplitude ~ RS^g)", (-1, 1)),
    ("gain_tf_a_tf", "TF shift a_tf (TF0 ~ RS^a_tf)", (-1.5, 1.5)),
    ("dR2_shift", "Held-out R², gain_tf - gain", None),
]):
    for (name, is_depth), color in zip([("depth selective", True), ("other", False)], GROUPS.values()):
        v = np.sort(hyp[col][hyp.depth_tuned == is_depth].dropna())
        ax.plot(v, np.arange(1, len(v) + 1) / len(v), color=color, lw=1.2, label=f"{name} (n = {len(v)})")
    ax.axvline(0, color="k", lw=0.5)
    if xlim is None:
        xlim = np.percentile(hyp[col].dropna(), [2, 98])
    ax.set_xlim(xlim)
    p = hyp_table.loc[HYP_MEASURES[col][0], "p_within_session_fit"]
    ax.set_title(f"depth > other: p = {p:.2g}", fontsize=6)
    ax.set_xlabel(xlabel, fontsize=6)
    ax.set_ylabel("Cumulative proportion", fontsize=6)
    ax.tick_params(labelsize=5)
    ax.ticklabel_format(axis="x", style="sci", scilimits=(-2, 2))
    ax.xaxis.get_offset_text().set_fontsize(5)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(fontsize=5, frameon=False)
fig.tight_layout()
style.savefig(SAVE_ROOT / "depth_cells_tf_shift.pdf", fig=fig, bbox_inches="tight")


### Which cells shift their preferred TF beyond gain? Model-free, per-cell test

`precompute_data/sftf_tf_shift.py` (results in `running_models/tf_shift.pickle`): for each SF/TF-tuned cell, the
running frames are split at the session's median running speed; TF curves of the slow and fast halves over the
SF / direction pairs that drive the cell (≥ 0.5 × best pair, same pairs for both halves); preferred TF of each half
= centre of mass in log2 TF of (curve − its minimum), which a multiplicative gain or an offset does not change;
shift = fast − slow (octaves). Null per cell: 200 surrogates from the cell's fitted `gain` model plus its own residuals
circularly shifted within each recording (noise structure and gain-induced SNR difference kept, no TF shift). A cell
"shifts" if its shift exceeds the surrogates (one-sided p < 0.05). Calibration (`tf_shift_sham.pickle`): the same with
the running-speed trace scrambled in time. Enrichment in depth-selective cells: permutation of the depth labels
within session, and within session × base-fit-quality tertile × preferred-TF tertile (depth-selective cells prefer
higher TF), 10000 permutations, one-sided.

**Relation to the population TF curves below** ("Population TF and SF tuning per running-speed bin"): both look at
TF curves per running speed over the same SF / direction pairs. The population curves average across cells, are
aligned on the full-fit preferred TF and normalised by each cell's maximum over all bins, so they keep the gain:
the asymmetric growth of the high-TF flank suggests a shift but can be distorted by gain and by averaging. This test
removes the gain per curve (each half normalised on its own), gives a statistic for every cell, and compares it to
a gain-only null, so it can say *which* cells shift and whether they are enriched in depth-selective cells.

In [ ]:
# per-cell TF shift beyond gain: enrichment in depth-selective cells, and calibration
from cottage_analysis.analysis import fit_gaussian_blob as fit_gb

shift_meta = rs_fits[["session", "roi", "depth_tuned", "any_sig", "base_test_rsq", "base_tf0", "base_log_amplitude",
                      "gain_g", "gain_tf_a_tf"]]
tf_shift = {
    name: pd.read_pickle(SAVE_ROOT / "running_models" / f"{name}.pickle").merge(shift_meta, on=["session", "roi"])
    for name in ["tf_shift", "tf_shift_sham"]
}
shift_rows = []
for name, x in tf_shift.items():
    x = x[np.isfinite(x.p_pos)].copy()  # flat curves (no shift) excluded
    x["shifter"] = x.p_pos < 0.05
    tf_shift[name] = x
    x["fit_tertile"] = pd.qcut(x.base_test_rsq, 3, labels=["low", "mid", "high"]).astype(str)
    x["tf_tertile"] = pd.qcut(x.base_tf0, 3, labels=["low", "mid", "high"]).astype(str)
    for cells_name, sub in [("all SF/TF tuned", x), ("selected", x[x.any_sig])]:
        _, p_s = depth_permutation(sub, "shifter", np.mean, ["session"])
        _, p_sft = depth_permutation(sub, "shifter", np.mean, ["session", "fit_tertile", "tf_tertile"])
        shift_rows.append(dict(
            analysis=name, cells=cells_name,
            depth=f"{sub.shifter[sub.depth_tuned].sum()}/{sub.depth_tuned.sum()} ({sub.shifter[sub.depth_tuned].mean():.1%})",
            other=f"{sub.shifter[~sub.depth_tuned].sum()}/{(~sub.depth_tuned).sum()} ({sub.shifter[~sub.depth_tuned].mean():.1%})",
            p_within_session=p_s, p_within_session_fit_tf=p_sft,
        ))
display(pd.DataFrame(shift_rows).set_index(["analysis", "cells"]))
shifts = tf_shift["tf_shift"]
print(f"median shift (octaves, fast - slow): observed {shifts['shift'].median():+.3f}, gain-only surrogates "
      f"{shifts.null_mean.median():+.3f}; Spearman with gain_tf a_tf {sstats.spearmanr(shifts['shift'], shifts.gain_tf_a_tf, nan_policy='omit')[0]:.2f}, "
      f"with gain g {sstats.spearmanr(shifts['shift'], shifts.gain_g, nan_policy='omit')[0]:.2f}")
shift_by_session = shifts.groupby(["session", "depth_tuned"]).shifter.mean().unstack()
display(shift_by_session.round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14 * CM, 5 * CM))
ax = axes[0]
for name, x, ls in [("data", tf_shift["tf_shift"], "-"), ("scrambled RS", tf_shift["tf_shift_sham"], ":")]:
    frac = x.groupby(["session", "depth_tuned"]).shifter.mean().unstack()
    for s in frac.index:
        ax.plot([0, 1], [frac[False].loc[s], frac[True].loc[s]], color="#8a8984" if ls == ":" else "k", lw=0.6, ls=ls, alpha=0.7)
    ax.scatter([0] * len(frac), frac[False], color=GROUPS["not depth tuned"], s=8, zorder=3)
    ax.scatter([1] * len(frac), frac[True], color=GROUPS["depth tuned"], s=8, zorder=3)
ax.axhline(0.05, color="k", lw=0.5, ls="--")
ax.set_xticks([0, 1], ["other", "depth selective"])
ax.set_xlim(-0.4, 1.4)
ax.set_ylabel("Fraction of cells with TF shift\n(beyond gain, p < 0.05)", fontsize=6)
ax.set_title("one line per session; dotted: scrambled RS", fontsize=6)
ax = axes[1]
for (name, is_depth), color in zip([("depth selective", True), ("other", False)], GROUPS.values()):
    v = np.sort(shifts["shift"][shifts.depth_tuned == is_depth].dropna())
    ax.plot(v, np.arange(1, len(v) + 1) / len(v), color=color, lw=1.2, label=f"{name} (n = {len(v)})")
ax.axvline(0, color="k", lw=0.5)
ax.set_xlim(-1.5, 1.5)
ax.set_xlabel("Preferred TF shift, fast - slow (octaves)", fontsize=6)
ax.set_ylabel("Cumulative proportion", fontsize=6)
ax.legend(fontsize=5, frameon=False)
for ax in axes:
    ax.tick_params(labelsize=5)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
style.savefig(SAVE_ROOT / "tf_shift_enrichment.pdf", fig=fig, bbox_inches="tight")


In [ ]:
# are the shifting cells particular? shifters vs non-shifters, within depth-selective and other cells
shape_rows = []
for s in SESSIONS:
    neurons = session_neurons(s)
    popt = neurons.rsof_popt_closedloop_g2d
    shape_rows.append(pd.DataFrame(dict(
        session=s, roi=neurons.roi.values,
        g2d_theta=popt.apply(fit_gb.get_gaussian_angle).values,
        g2d_log2_elongation=np.log2(popt.apply(fit_gb.get_semimajor_length) / popt.apply(fit_gb.get_semiminor_length)).values,
        log10_preferred_rs=np.log10(neurons.preferred_RS_closedloop_g2d.astype(float).values),
        log10_preferred_of=np.log10(np.degrees(neurons.preferred_OF_closedloop_g2d.astype(float).values)),
        log10_preferred_depth=np.log10(neurons.preferred_depth_closedloop.astype(float).values),
    )))
char = shifts.merge(pd.concat(shape_rows), on=["session", "roi"]).merge(
    cells[["session", "roi", "gosi", "gdsi"]], on=["session", "roi"], how="left")
char["g2d_dist_to_diagonal"] = (char.g2d_theta - 45).abs()
FEATURES = ["log10_preferred_depth", "log10_preferred_rs", "log10_preferred_of", "g2d_dist_to_diagonal",
            "g2d_log2_elongation", "base_tf0", "gosi", "gdsi", "gain_g", "base_log_amplitude", "base_test_rsq"]


def shifter_permutation(df, col, n=N_PERM):
    # median difference shifters - others and two-sided p, shifter labels permuted within session x fit tertile
    df = df.dropna(subset=[col])
    vals, labels = df[col].to_numpy(float), df.shifter.to_numpy(bool)
    obs = np.median(vals[labels]) - np.median(vals[~labels])
    keys = (df.session + "_" + df.fit_tertile).to_numpy()
    groups = [np.flatnonzero(keys == k) for k in np.unique(keys)]
    null = np.empty(n)
    for i in range(n):
        lab = labels.copy()
        for idx in groups:
            lab[idx] = perm_rng.permutation(lab[idx])
        null[i] = np.median(vals[lab]) - np.median(vals[~lab])
    return obs, (1 + np.sum(np.abs(null) >= abs(obs))) / (n + 1)


char_rows = []
for group, sub in [("depth selective", char[char.depth_tuned]), ("other", char[~char.depth_tuned])]:
    for f in FEATURES:
        diff, p_ctrl = shifter_permutation(sub, f)
        char_rows.append(dict(group=group, feature=f, shifters=sub[f][sub.shifter].median(),
                              others=sub[f][~sub.shifter].median(),
                              p_mwu=sstats.mannwhitneyu(sub[f][sub.shifter].dropna(), sub[f][~sub.shifter].dropna()).pvalue,
                              p_within_session_fit=p_ctrl))
display(pd.DataFrame(char_rows).set_index(["group", "feature"]).round(3))


In [ ]:
# examples: depth-selective shifters and non-shifters with strong responses; slow vs fast TF curves
FRAMES_DIR = SAVE_ROOT / "running_models" / "frames"
ex_pool = shifts[shifts.depth_tuned & (shifts.base_log_amplitude > np.log(0.5))]
ex_cells = pd.concat([
    ex_pool[ex_pool.shifter & np.isfinite(ex_pool["shift"])].sort_values(["p_pos", "base_log_amplitude"], ascending=[True, False]).head(3).assign(kind="shifter"),
    ex_pool[ex_pool.p_pos > 0.5].sort_values("base_test_rsq", ascending=False).head(2).assign(kind="non-shifter"),
])
fig, axes = plt.subplots(2, len(ex_cells), figsize=(18 * CM, 7.5 * CM))
for j, (_, cell) in enumerate(ex_cells.iterrows()):
    d = np.load(FRAMES_DIR / f"{cell.session}.npz")
    col = int(np.flatnonzero(d["rois"] == cell.roi)[0])
    rs_cm_s = np.exp(d["X"][3] + float(d["log_rs_median"])) * 100
    split = [rs_cm_s.min() * 0.999, np.median(rs_cm_s), rs_cm_s.max() * 1.001]
    curves, tf, labels = sftf_examples.population_tf_curves(d["X"], d["dff"][:, [col]], float(d["log_rs_median"]), split)
    for i, norm in enumerate([False, True]):
        ax = axes[i, j]
        for b, color in zip(range(2), ["#8a8984", "k"]):
            c = curves[0, b]
            if norm:  # gain removed as in the test: minimum subtracted, normalised to sum 1
                c = (c - np.nanmin(c)) / np.nansum(c - np.nanmin(c))
            ax.plot(tf, c, color=color, lw=1, marker="o", ms=2, label=["slow half", "fast half"][b])
            if norm:
                ax.axvline(2 ** sftf_examples.tf_centre_of_mass(curves[0, b], tf), color=color, lw=0.6, ls=":")
        ax.set_xscale("log")
        ax.set_xticks(tf, [f"{t:g}" for t in tf])
        ax.minorticks_off()
        ax.tick_params(labelsize=5)
        ax.spines[["top", "right"]].set_visible(False)
        if i == 0:
            ax.set_title(f"{cell.kind}\n{cell.session.split('_')[1]} ROI {cell.roi}\nshift {cell['shift']:+.2f} oct, p = {cell.p_pos:.3g}", fontsize=5.5)
        else:
            ax.set_xlabel("TF (Hz)", fontsize=6)
    axes[0, 0].set_ylabel("ΔF/F", fontsize=6)
    axes[1, 0].set_ylabel("Normalised\n(gain removed)", fontsize=6)
axes[0, -1].legend(fontsize=5, frameon=False)
fig.tight_layout()
style.savefig(SAVE_ROOT / "tf_shift_examples.pdf", fig=fig, bbox_inches="tight")


### TF × RS slope against the shape of the RS/OF tuning (as figure 3K)

Polar plot of figure 3K (`figure_utils.rsof_integration.plot_angle_eccentricity_polar`): orientation of the RS/OF
2D Gaussian (`rsof_popt_closedloop_g2d`) against log2 elongation (σ_major / σ_minor), coloured by the TF × RS slope
(`tf_a_tf`, single-slope model). Unlike figure 3K (treadmill, trial-average fits), the RS/OF fits here are the
per-frame free-locomotion fits of these sessions. Depth-selective cells only, with an RS/OF fit above its
empirical-null threshold, among the cells of the running-speed fits; left all of them, right those also
significant in a running-speed model. Black edges show the points whose slope is near 0 (white).
Grey: elongation ≤ 1.4 (orientation undefined, as in figure 3K).

In [ ]:
from cottage_analysis.analysis import fit_gaussian_blob as fit_gb
from v1_depth_map.figure_utils.rsof_integration import plot_angle_eccentricity_polar

ELONGATION_CUTOFF = 1.4  # as figure 3K
SLOPE_LIM = 1.0  # colour scale of the TF x RS slope, +- this


def g2d_shape(session):
    popt = session_neurons(session).rsof_popt_closedloop_g2d
    return pd.DataFrame(dict(
        session=session, roi=np.arange(len(popt)),
        g2d_theta=popt.apply(fit_gb.get_gaussian_angle).values,
        g2d_elongation=(popt.apply(fit_gb.get_semimajor_length) / popt.apply(fit_gb.get_semiminor_length)).values,
    ))


shape = pd.concat([g2d_shape(s) for s in SESSIONS], ignore_index=True)
polar_df = rs_fits.merge(shape, on=["session", "roi"], how="left").merge(
    cells[["session", "roi", "rsof_tuned"]], on=["session", "roi"], how="left"
)
polar_depth = polar_df[polar_df.depth_tuned.astype(bool)]
polar_sets = {
    "RS/OF fit sig.": polar_depth[polar_depth.rsof_tuned],
    "RS/OF and running-speed fits sig.": polar_depth[polar_depth.rsof_tuned & polar_depth.any_sig],
}
fig, axes = plt.subplots(1, 2, figsize=(16 * CM, 8 * CM), subplot_kw=dict(projection="polar"))
fig.subplots_adjust(wspace=0.45)
for ax, (name, d) in zip(axes, polar_sets.items()):
    elong_ok = d.g2d_elongation > ELONGATION_CUTOFF
    sc = plot_angle_eccentricity_polar(
        ax, d.loc[elong_ok, "g2d_theta"], None, FONTSIZE_DICT_SMALL, radial_scale="elongation",
        axis_ratio=d.loc[elong_ok, "g2d_elongation"], scale=0.5, rasterize_schematics="each",
        c=d.loc[elong_ok, "tf_a_tf"], cmap="RdBu_r", vmin=-SLOPE_LIM, vmax=SLOPE_LIM, s=12, alpha=0.9,
        edgecolors="k", linewidths=0.3,
    )
    ax.scatter(np.radians(d.loc[~elong_ok, "g2d_theta"]), np.clip(np.log2(d.loc[~elong_ok, "g2d_elongation"]), 0, None),
               c="#c3c2b7", s=8, linewidths=0, zorder=2)
    rho, p = stats.spearmanr(np.abs(d.loc[elong_ok, "g2d_theta"] - 45), d.loc[elong_ok, "tf_a_tf"])
    ax.set_title(f"{name}, n = {elong_ok.sum()} (+{(~elong_ok).sum()} grey)\n"
                 f"|θ - 45°| vs slope: ρ = {rho:.2f}, p = {p:.2g}", fontsize=6, pad=14)
cbar = fig.colorbar(sc, ax=axes, shrink=0.6, pad=0.08)
cbar.set_label("TF × RS slope (log TF per log RS)", fontsize=6)
cbar.ax.tick_params(labelsize=5)
style.savefig(SAVE_ROOT / "polar_tf_slope.pdf", fig=fig, bbox_inches="tight")

### Population TF and SF tuning per running-speed bin

TF curves per speed bin (as the example curves above: SF / direction pairs ≥ `TF_SELECT_THRESHOLD` of the
best pair, same pairs for all bins), for the cells of the model comparison (significant in at least one model)
whose preferred TF in the full-data `base` fit is inside the tested range (0.5-16 Hz). TF expressed in octaves
from that preferred TF (log2 TF / TF0, binned to whole octaves); each cell's curves normalised by its maximum over
all bins (gain differences between bins are kept). Mean ± SEM across cells; points with < 10 cells not shown.
Bottom row: the same for SF (TF / direction pairs ≥ threshold, SF in octaves from the preferred SF of the base fit,
cells with preferred SF in 0.01-0.32 cpd).

In [ ]:
FRAMES_DIR = SAVE_ROOT / "running_models" / "frames"
MIN_CELLS_PER_POINT = 10
POP_VARIABLES = {  # variable: (label, tested range of the preference)
    "tf": ("TF", (0.5, 16)),
    "sf": ("SF", (0.01, 0.32)),
}
pop_means, pop_n = {}, {}
for var, (var_label, (lo, hi)) in POP_VARIABLES.items():
    pop_cells = rs_fits[rs_fits.any_sig & rs_fits[f"base_{var}0"].between(np.log(lo), np.log(hi))]
    pop = []
    for session, sc in pop_cells.groupby("session"):
        d = np.load(FRAMES_DIR / f"{session}.npz")
        cols = [int(np.flatnonzero(d["rois"] == r)[0]) for r in sc.roi]
        curves, values, speed_labels = sftf_examples.population_tf_curves(
            d["X"], d["dff"][:, cols], float(d["log_rs_median"]), SPEED_EDGES_CM_S,
            threshold=TF_SELECT_THRESHOLD, variable=var,
        )
        norm = np.nanmax(curves, axis=(1, 2), keepdims=True)
        for k, (_, cell) in enumerate(sc.iterrows()):
            delta = np.round(np.log2(values) - cell[f"base_{var}0"] / np.log(2)).astype(int)
            for b, label in enumerate(speed_labels):
                pop.append(pd.DataFrame(dict(session=session, roi=cell.roi, depth_tuned=cell.depth_tuned, speed_bin=b,
                                             speed=label, delta=delta, resp=curves[k, b] / norm[k, 0, 0])))
    pop = pd.concat(pop, ignore_index=True)
    # average within cell first (two values can fall in the same octave bin), then across cells
    pop_cell = pop.groupby(["depth_tuned", "speed_bin", "speed", "delta", "session", "roi"]).resp.mean().reset_index()
    pop_means[var] = pop_cell.groupby(["depth_tuned", "speed_bin", "speed", "delta"]).resp.agg(["mean", "sem", "count"]).reset_index()
    pop_n[var] = pop_cells.depth_tuned.value_counts()
    print(f"{var_label}: {len(pop_cells)} cells, {pop_cells.depth_tuned.sum()} depth selective")

fig, axes = plt.subplots(2, 2, figsize=(14 * CM, 10 * CM), sharey=True)
colors = plt.get_cmap("viridis")(np.linspace(0, 0.9, len(speed_labels)))
for i, (var, (var_label, _)) in enumerate(POP_VARIABLES.items()):
    for j, (name, is_depth) in enumerate([("depth selective", True), ("not depth selective", False)]):
        ax = axes[i, j]
        m = pop_means[var]
        sub = m[(m.depth_tuned == is_depth) & (m["count"] >= MIN_CELLS_PER_POINT)]
        for b, (label, color) in enumerate(zip(speed_labels, colors)):
            c = sub[sub.speed_bin == b].sort_values("delta")
            ax.errorbar(c.delta, c["mean"], yerr=c["sem"], color=color, lw=1, marker="o", ms=2, capsize=0, label=label)
        ax.set_title(f"{var_label}, {name} (n = {pop_n[var].get(is_depth, 0)})", fontsize=6)
        ax.axvline(0, color="k", lw=0.5, ls=":")
        ax.set_xlabel(f"{var_label} relative to preferred {var_label} (octaves)", fontsize=6)
        ax.tick_params(labelsize=5)
        ax.spines[["top", "right"]].set_visible(False)
    axes[i, 0].set_ylabel("Normalised ΔF/F", fontsize=6)
axes[0, 1].legend(title="Running speed", fontsize=5, title_fontsize=5, frameon=False, bbox_to_anchor=(1, 1), loc="upper left")
fig.tight_layout()
style.savefig(SAVE_ROOT / "population_tf_curves_by_speed.pdf", fig=fig, bbox_inches="tight")

## Exploratory: SF / TF preferences vs depth and speed preferences

Preferred SF, TF and grating speed (TF / SF) against preferred virtual depth (depth-selective, SF/TF-tuned
cells) and against the preferred optic flow and running speed of the RS/OF fit (SF/TF- and RS/OF-tuned
cells, as in B and C). Points coloured by local density. Spearman ρ across cells with a hierarchical bootstrap
(mice, sessions, `N_BOOTS_EXPLORATORY` resamples). Not exported to the manuscript stats.

In [ ]:
# exploratory correlations
EXPLORE_X = {  # column: (label, transform)
    "fit_sf0": ("Preferred SF (cpd)", np.exp),
    "fit_tf0": ("Preferred TF (Hz)", np.exp),
    "pref_speed": ("Preferred grating speed (deg/s)", lambda v: v),
}
EXPLORE_Y = {  # column: (label, transform, selection)
    "preferred_depth": ("Preferred virtual depth (cm)", lambda v: v * 100, cells.depth_tuned & cells.sftf_tuned),
    "preferred_of": ("Preferred optic flow (deg/s)", lambda v: v, cells.sftf_tuned & cells.rsof_tuned),
    "preferred_rs": ("Preferred running speed (cm/s)", lambda v: v * 100, cells.sftf_tuned & cells.rsof_tuned),
}
explore = {}
for ycol, (_, yf, sel) in EXPLORE_Y.items():
    sub = cells[sel]
    for xcol, (_, xf) in EXPLORE_X.items():
        st = bootstrap_spearman(xf(sub[xcol]), yf(sub[ycol]), sub.session, n_boots=N_BOOTS_EXPLORATORY)
        explore[(xcol, ycol)] = st
        print(f"{xcol:10s} vs {ycol:15s}: n = {st['n']}, rho = {st['rho']:.2f}, bootstrap p = {st['p']:.2g}")

In [ ]:
from cottage_analysis.plotting import plotting_utils

fig, axes = plt.subplots(len(EXPLORE_Y), len(EXPLORE_X), figsize=(17 * CM, 15 * CM))
for i, (ycol, (ylabel, yf, sel)) in enumerate(EXPLORE_Y.items()):
    sub = cells[sel]
    for j, (xcol, (xlabel, xf)) in enumerate(EXPLORE_X.items()):
        ax, st = axes[i, j], explore[(xcol, ycol)]
        plotting_utils.density_scatter(xf(sub[xcol]), yf(sub[ycol]), ax=ax, log_scale=True, bins=32, s=5)
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.text(0.5, 1.03, f"ρ = {st['rho']:.2f} (n = {st['n']}), ", transform=ax.transAxes, ha="right",
                va="bottom", fontsize=6)
        ax.text(0.5, 1.03, f"p = {st['p']:.2g}", transform=ax.transAxes, ha="left", va="bottom", fontsize=6,
                color="green" if st["p"] < 0.05 else "red")
        if i == len(EXPLORE_Y) - 1:
            ax.set_xlabel(xlabel)
        if j == 0:
            ax.set_ylabel(ylabel)
        ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
style.savefig(SAVE_ROOT / "exploratory_sftf_correlations.pdf", fig=fig, bbox_inches="tight")

## Proportions of responsive and selective cells

In [ ]:
# proportions per group and χ² depth tuned vs not
SELECTIVITY = ["responsive", "ori_selective", "dir_selective"]
counts = cells.groupby("group")[SELECTIVITY].agg(["sum", "mean"])
counts.insert(0, "n", cells.groupby("group").size())
display(counts)

per_session = cells.groupby(["session", "group"])[SELECTIVITY].mean().unstack("group")
display(per_session.round(2))

prop_tests = {}
for col in SELECTIVITY:
    chi2, p, _, _ = stats.chi2_contingency(pd.crosstab(cells.group, cells[col]))
    prop_tests[col] = dict(chi2=chi2, p=p)
    print(f"{col:14s} depth tuned vs not: chi2 = {chi2:.1f}, p = {p:.2g}")

In [ ]:
labels = ["responsive", "orientation\nselective", "direction\nselective"]
fig, ax = plt.subplots(figsize=(8 * CM, 5 * CM))
width = 0.38
for i, (group, color) in enumerate(GROUPS.items()):
    x = np.arange(len(SELECTIVITY)) + (i - 0.5) * width
    ax.bar(x, counts.loc[group, [(c, "mean") for c in SELECTIVITY]].values, width=width * 0.92,
           color=color, label=group)
    sess = per_session.xs(group, axis=1, level="group")[SELECTIVITY]
    for j, c in enumerate(SELECTIVITY):
        ax.scatter(np.full(len(sess), x[j]), sess[c], s=6, color="k", zorder=3, linewidths=0)
ax.set_xticks(np.arange(len(SELECTIVITY)), labels)
ax.set_ylabel("Proportion of cells")
ax.set_ylim(0, 1)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
style.savefig(SAVE_ROOT / "proportions.pdf", fig=fig, bbox_inches="tight")

## Preferred orientation and direction

Orientation preference of orientation-selective cells and direction preference of direction-selective
cells, one bin per stimulus orientation / direction (the vector sum is continuous, but only 8 directions
were shown). Comparison: χ² on the binned counts.

In [ ]:
# binned preferences per group and χ² depth tuned vs not
def binned_counts(values, period, n_bins):
    # counts in bins centred on multiples of period / n_bins
    width = period / n_bins
    edges = np.arange(n_bins + 1) * width - width / 2
    return np.histogram((values + width / 2) % period - width / 2, bins=edges)[0], edges


PREFERENCES = {
    "orientation": dict(col="pref_ori", sel="ori_selective", period=180, n_bins=4),
    "direction": dict(col="pref_dir", sel="dir_selective", period=360, n_bins=8),
}
pref_counts, pref_tests = {}, {}
for name, d in PREFERENCES.items():
    pref_counts[name] = {
        group: binned_counts(cells.loc[(cells.group == group) & cells[d["sel"]], d["col"]].values,
                             d["period"], d["n_bins"])
        for group in GROUPS
    }
    table = np.array([c for c, _ in pref_counts[name].values()])
    chi2, p, _, _ = stats.chi2_contingency(table + 0.5)
    pref_tests[name] = dict(chi2=chi2, p=p, n=int(table.sum()))
    print(f"preferred {name}: {dict(zip(GROUPS, table.tolist()))}, chi2 p = {p:.2g}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16 * CM, 5 * CM))
for ax, (name, d) in zip(axes, PREFERENCES.items()):
    for group, color in GROUPS.items():
        c, edges = pref_counts[name][group]
        ax.step(np.append(edges[:-1], edges[-1]), np.append(c, c[-1]) / c.sum(),
                where="post", color=color, lw=1.5, label=f"{group} (n={c.sum()})")
    ax.set_title(f"Preferred {name} ({d['sel'].replace('_', ' ')})\nχ² p = {pref_tests[name]['p']:.2g}",
                 fontsize=7)
    ax.set_xticks(np.arange(0, d["period"] + 1, 45))
    ax.set_ylim(0, None)
    ax.set_xlabel(f"Preferred {name} (deg)")
    ax.set_ylabel("Proportion of cells")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False, fontsize=6)
style.savefig(SAVE_ROOT / "preferred_orientation_direction.pdf", fig=fig, bbox_inches="tight")

## Selectivity indices

gOSI and gDSI of responsive cells (Mann-Whitney U).

In [ ]:
# gOSI / gDSI of responsive cells per group, Mann-Whitney U depth tuned vs not
index_samples, index_tests = {}, {}
for col in ["gosi", "gdsi"]:
    index_samples[col] = {
        group: np.sort(cells.loc[(cells.group == group) & cells.responsive, col].dropna().values)
        for group in GROUPS
    }
    a, b = index_samples[col].values()
    index_tests[col] = dict(p=stats.mannwhitneyu(a, b).pvalue, median=[np.median(a), np.median(b)])
    print(f"{col}: medians {np.round(index_tests[col]['median'], 3)}, p = {index_tests[col]['p']:.2g}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12 * CM, 5 * CM))
for ax, col in zip(axes, ["gosi", "gdsi"]):
    for group, color in GROUPS.items():
        v = index_samples[col][group]
        ax.plot(v, np.arange(1, len(v) + 1) / len(v), color=color, lw=1.5, label=f"{group} (n={len(v)})")
    ax.set_title(f"{col} (responsive cells), p = {index_tests[col]['p']:.2g}", fontsize=7)
    ax.set_xlabel(col)
    ax.set_ylabel("Cumulative proportion")
    ax.set_xlim(0, 1)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False, fontsize=6)
style.savefig(SAVE_ROOT / "selectivity_indices.pdf", fig=fig, bbox_inches="tight")

## Preferred orientation vs preferred depth

Depth-tuned, orientation-selective cells. Independent preferences would show as no structure here.

In [ ]:
# preferred depth across the 4 orientation bins (centred on the stimulus orientations), Kruskal-Wallis
ori_depth = cells[cells.depth_tuned & cells.ori_selective]
ori_bin = ((ori_depth.pref_ori + 22.5) % 180 // 45 * 45).astype(int)
kw = stats.kruskal(*[np.log(ori_depth.preferred_depth[ori_bin == b]) for b in sorted(ori_bin.unique())])
ori_depth_test = dict(n=len(ori_depth), H=kw.statistic, p=kw.pvalue)
print(ori_depth_test)

In [ ]:
fig, ax = plt.subplots(figsize=(6 * CM, 5 * CM))
ax.scatter(ori_depth.preferred_depth, ori_depth.pref_ori, s=6, color=GROUPS["depth tuned"], linewidths=0)
ax.set_xscale("log")
ax.set_xlabel("Preferred virtual depth (m)")
ax.set_ylabel("Preferred orientation (deg)")
ax.set_yticks(np.arange(0, 181, 45))
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(f"n = {ori_depth_test['n']}, Kruskal-Wallis across orientation bins p = {ori_depth_test['p']:.2g}",
             fontsize=7)
style.savefig(SAVE_ROOT / "orientation_vs_depth.pdf", fig=fig, bbox_inches="tight")

## Export statistics

Every value is read from a variable computed above for a panel; nothing is recomputed. Written to
`v1_depth_map/stats/generated/stats_fig_reviewer_gratings.yaml` (group `reviewer`, after the manuscript
figures); `python -m v1_depth_map.stats.build_manuscript_stats` compiles it into the dashboard and the
LaTeX macros.

In [ ]:
from v1_depth_map.stats import export_figure_stats


def _fmt_pval(p):
    # p-value as it should read in LaTeX, in math mode so it works in text or math
    p = float(p)
    return "$p < 0.0001$" if p < 1e-4 else f"$p = {p:.3g}$"


def _stat(raw, formatted, macro, description):
    return dict(raw=raw, formatted=formatted, latex_macro=macro, description=description)


def _count(n, macro, description):
    return _stat(int(n), f"{int(n):,}", macro, description)


def _pct(fraction, macro, description):
    return _stat(float(100 * fraction), f"{100 * fraction:.0f}\\%", macro, description)


def _p(p, macro, description):
    return _stat(float(p), _fmt_pval(p), macro, description)


def _rho(r, macro, description):
    return _stat(float(r), f"{r:.2f}", macro, description)


n_all = venn["n_all"]
groups_n = cells.group.value_counts()
stats_reviewer = {
    "n_neurons": _count(n_all, "statRevGratNeurons", "iscell neurons in the 6 grating sessions"),
    "n_sessions": _count(cells.session.nunique(), "statRevGratSessions", "Sessions with drifting gratings"),
    "n_mice": _count(cells.session.str.split("_").str[0].nunique(), "statRevGratMice", "Mice with drifting gratings"),
    "n_depth": _count(venn["n_depth"], "statRevGratDepthNeurons", "Depth-selective neurons (A)"),
    "pct_depth": _pct(venn["n_depth"] / n_all, "statRevGratPctDepth", "Percentage depth selective (A)"),
    "n_sftf": _count(venn["n_sftf"], "statRevGratSftfNeurons", "SF/TF-tuned neurons, cross-validated (A)"),
    "pct_sftf": _pct(venn["n_sftf"] / n_all, "statRevGratPctSftf", "Percentage SF/TF tuned (A)"),
    "pct_both": _pct(venn["n_both"] / n_all, "statRevGratPctBoth", "Percentage depth selective and SF/TF tuned (A)"),
    "pct_both_expected": _pct(venn["n_both_expected"] / n_all, "statRevGratPctBothExpected",
                              "Percentage both expected if independent (A)"),
    "rsof_threshold": _stat(float(rsof_thresholds["g2d"]), f"{rsof_thresholds['g2d']:.4f}", "statRevGratRsofThreshold",
                            "RS/OF test R² empirical-null threshold (95th percentile, as figure 2)"),
    "speed_of_n": _count(stat_speed_of["n"], "statRevGratSpeedOfN", "SF/TF- and RS/OF-tuned neurons (B)"),
    "speed_of_rho": _rho(stat_speed_of["rho"], "statRevGratSpeedOfRho",
                         "Spearman rho, preferred grating speed vs preferred optic flow (B)"),
    "speed_of_p": _p(stat_speed_of["p"], "statRevGratSpeedOfP", "Hierarchical bootstrap p (B)"),
    "speed_depth_rho": _rho(stat_speed_depth["rho"], "statRevGratSpeedDepthRho",
                            "Spearman rho, preferred grating speed vs preferred depth (C)"),
    "speed_depth_p": _p(stat_speed_depth["p"], "statRevGratSpeedDepthP", "Hierarchical bootstrap p (C)"),
    "n_tuned_depth": _count(len(tuned), "statRevGratTunedDepthN", "SF/TF-tuned depth-selective neurons (D, E)"),
    "n_dir_tuned": _count(len(dir_tuned), "statRevGratDirTunedN", f"Of which dsi > {DSI_THR} (F)"),
}
for col, macro in [("responsive", "Resp"), ("ori_selective", "OriSel"), ("dir_selective", "DirSel")]:
    stats_reviewer[f"{col}_pct_depth"] = _pct(
        counts.loc["depth tuned", (col, "mean")], f"statRevGrat{macro}PctDepth", f"Percentage {col}, depth tuned")
    stats_reviewer[f"{col}_pct_other"] = _pct(
        counts.loc["not depth tuned", (col, "mean")], f"statRevGrat{macro}PctOther", f"Percentage {col}, not depth tuned")
    stats_reviewer[f"{col}_p"] = _p(prop_tests[col]["p"], f"statRevGrat{macro}P", f"chi2 p, {col} depth tuned vs not")
stats_reviewer["pref_ori_p"] = _p(pref_tests["orientation"]["p"], "statRevGratPrefOriP",
                                  "chi2 p, preferred orientation depth tuned vs not")
stats_reviewer["pref_dir_p"] = _p(pref_tests["direction"]["p"], "statRevGratPrefDirP",
                                  "chi2 p, preferred direction depth tuned vs not")
stats_reviewer["gosi_p"] = _p(index_tests["gosi"]["p"], "statRevGratGosiP", "Mann-Whitney p, gOSI depth tuned vs not")
stats_reviewer["gdsi_p"] = _p(index_tests["gdsi"]["p"], "statRevGratGdsiP", "Mann-Whitney p, gDSI depth tuned vs not")
stats_reviewer["ori_depth_n"] = _count(ori_depth_test["n"], "statRevGratOriDepthN",
                                       "Depth-tuned orientation-selective neurons")
stats_reviewer["ori_depth_p"] = _p(ori_depth_test["p"], "statRevGratOriDepthP",
                                   "Kruskal-Wallis p, preferred depth across orientation bins")

export_figure_stats("fig_reviewer_gratings", stats_reviewer, figure="reviewer")